In [ ]:
# Imports

import numpy as np
import os

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, accuracy_score
 
import tensorflow as tf
import tensorflow_model_optimization as tfmot

keras = tf.keras
layers = tf.keras.layers

In [ ]:
def quantize_and_save(model, export_dir, model_name, quantization_type='int8', representative_data=None):
    """
    Quantizes a Keras model to TFLite format.
    Types: 'float16', 'int8' (dynamic), 'full_int8'
    """
    
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    
    if quantization_type == 'float16':
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]
        
    elif quantization_type == 'int8':
        # Dynamic range quantization (weights to int8, activations float)
        converter.optimizations = [tf.lite.Optimize.DEFAULT]

    elif quantization_type == 'int16':
        # Integer quantization with int16 activations and int8 weights
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.int16]
        
    elif quantization_type == 'full_int8':
        # Full integer quantization (requires representative dataset)
        if representative_data is None:
            raise ValueError("Full integer quantization requires representative_data.")
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = representative_data
        # Ensure fallback to float if an op isn't supported in int8
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.int8
        converter.inference_output_type = tf.int8

    tflite_model = converter.convert()
    
    # Save the model
    save_path = f"{export_dir}/{model_name}_{quantization_type}.tflite"
    with open(save_path, "wb") as f:
        f.write(tflite_model)
    
    print(f"Quantized model saved to: {save_path}")
    return tflite_model

In [ ]:
# Representative data generator, with take into account the imbalance data
# Needed for full integer quantization

def representative_data_gen():
    # Number of samples per class to ensure diversity
    samples_per_class = 20 
    unique_classes = np.unique(y_tr)
    
    representative_indices = []
    
    for cls in unique_classes:
        # Find indices where the label matches the class
        indices = np.where(y_tr == cls)[0]
        # Randomly pick 20 samples from this class
        selected = np.random.choice(indices, samples_per_class, replace=False)
        representative_indices.extend(selected)
    
    # Shuffle the final selection
    np.random.shuffle(representative_indices)
    
    for idx in representative_indices:
        yield [
            Xe_tr[idx:idx+1].astype(np.float32), 
            Xr_tr[idx:idx+1].astype(np.float32)
        ]

In [ ]:
def evaluate_tflite_model(tflite_path, Xe_test, Xr_test, y_test):
    # Load the TFLite model and allocate tensors
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()

    # Get input and output details
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    y_pred = []

    print(f"Evaluating {tflite_path}...")
    for i in range(len(Xe_test)):
        # Prepare inputs (must match the order in input_details)
        # Check input_details[0]['name'] if you aren't sure of the order
        interpreter.set_tensor(input_details[0]['index'], Xe_test[i:i+1].astype(np.float32))
        interpreter.set_tensor(input_details[1]['index'], Xr_test[i:i+1].astype(np.float32))

        # Run inference
        interpreter.invoke()

        # Get the result
        output_data = interpreter.get_tensor(output_details[0]['index'])
        y_pred.append(np.argmax(output_data))

    # Calculate metrics
    acc = accuracy_score(y_test, y_pred)
    print(f"\nAccuracy: {acc*100:.2f}%")
    print(classification_report(y_test, y_pred, target_names=['N', 'S', 'V', 'F', 'Q']))

### Main usage

In [ ]:
# Load saved ECG MIT-BIH Arrhythmia dataset
data = np.load("processed_mit_bih_arrhythmia_dataset.npz")

# Extract the arrays back into variables
Xe_tr, Xr_tr, y_tr = data['Xe_tr'], data['Xr_tr'], data['y_tr']
Xe_vl, Xr_vl, y_vl = data['Xe_vl'], data['Xr_vl'], data['y_vl']
Xe_te, Xr_te, y_te = data['Xe_te'], data['Xr_te'], data['y_te']

print("Data loaded successfully.")
print(f"Train shapes: {Xe_tr.shape}, {Xr_tr.shape}")

In [ ]:
# Load the model to quantize
export_dir = 'ecg_classifier_quantized_models'
model_name_list = ['conv1d_t_model', 'conv1d_2d_t_model']
quantization_types = ['float16', 'int16', 'int8'] # It could be float16, int16, int8 or full_int8 quantization

for model_name in model_name_list:
    for quantization_type in quantization_types:
        model_to_quantize = tf.keras.models.load_model(f'stft_cnn_ecg_classifier_models/{model_name}.keras')
        quantize_and_save(model=model_to_quantize, export_dir=export_dir, model_name=model_name, quantization_type=quantization_type)